In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import requests
from pysus import sim


In [9]:
codigos_x = [f"X{i}" for i in range(85, 100)]
codigo_y = [f"Y0{i}" for i in range(0, 10)]
codigo_agressao = codigos_x + codigo_y


In [10]:
estados = ["AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", 
        "MA", "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", 
        "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO"]

anos = list(range(2015, 2022))

if os.path.exists("violencia_feminina.parquet"):    
    violencia_feminina = pd.read_parquet("violencia_feminina.parquet")
    print("Dados carregados do arquivo parquet.")
else:
    dfs = []
    
    for estado in estados:
        for ano in anos:
            df_temp = sim(state=estado, year=ano)
            df_filtrado = df_temp[
                (df_temp["SEXO"] == "2") &
                (df_temp["CAUSABAS"].str[:3].isin(codigo_agressao))
            ]
            dfs.append(df_filtrado)
            del df_temp
                
    violencia_feminina = pd.concat(dfs, ignore_index=True)
    violencia_feminina.to_parquet("violencia_feminina.parquet")

violencia_feminina["ANO"] = pd.to_datetime(
violencia_feminina["DTOBITO"], format="%d%m%Y", errors="coerce"
    ).dt.year
violencia_feminina = violencia_feminina.dropna(subset=["ANO"])
violencia_feminina["ANO"] = violencia_feminina["ANO"].astype(int)

print(f"Total de registros: {violencia_feminina.shape[0]}")
print(f"total de colunas: {violencia_feminina.shape[1]}")


Dados carregados do arquivo parquet.
Total de registros: 33873
total de colunas: 89


In [11]:
violencia_feminina["ESTADO"] = violencia_feminina["CODMUNOCOR"].str.strip().str[:2]

codigo_estado = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA",
    "16": "AP", "17": "TO", "21": "MA", "22": "PI", "23": "CE",
    "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE",
    "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT",
    "52": "GO", "53": "DF"
}

violencia_feminina["ESTADO"] = violencia_feminina["ESTADO"].map(codigo_estado)


In [12]:
url = "https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2015|2016|2017|2018|2019|2020|2021|2022/variaveis/9324?localidades=N3[all]"

response = requests.get(url)
dados = response.json()
print(response.status_code)

200


In [13]:
registros = []

for serie in dados[0]["resultados"][0]["series"]:
    estado = codigo_estado[serie["localidade"]["id"]]
    for ano, populacao in serie["serie"].items():
        registros.append({
            "ESTADO": estado,
            "ANO": int(ano),
            "POPULACAO": int(populacao)
        })
        

populacao_df = pd.DataFrame(registros)
print(populacao_df.head(5))
print(populacao_df.shape)


  ESTADO   ANO  POPULACAO
0     RO  2015    1768204
1     RO  2016    1787279
2     RO  2017    1805788
3     RO  2018    1757589
4     RO  2019    1777225
(189, 3)


In [14]:
url = "https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2015|2016|2017|2018|2019|2020|2021|2022/variaveis/9324?localidades=N3[all]"

response = requests.get(url)
dados = response.json()


In [15]:
violencia_feminina.head(10)

,CONTADOR,ORIGEM,TIPOBITO,DTOBITO,HORAOBITO,NATURAL,CODMUNNATU,DTNASC,IDADE,SEXO,...,TPRESGINFO,TPNIVELINV,NUDIASINF,DTCADINF,MORTEPARTO,DTCONCASO,FONTESINF,ALTCAUSA,ANO,ESTADO
0,212,1,2,13022015,1515,812,120030,27061989,425,2,...,,M,,,,,,,2015,AC
1,265,1,2,10012015,1800,812,120020,27091987,427,2,...,,M,,,,,,,2015,AC
2,328,1,2,29032015,,812,120060,09091998,416,2,...,,M,,,,,,,2015,AC
3,519,1,2,20012015,2200,812,120040,28051998,416,2,...,,M,,,,,,,2015,AC
4,925,1,2,02032015,2335,812,120040,07021986,429,2,...,,M,,,,,,,2015,AC
5,1077,1,2,22102015,1700,812,120035,04101987,428,2,...,,M,,,,,,,2015,AC
6,1702,1,2,13052015,2300,812,120050,13111980,434,2,...,,M,,,,,,,2015,AC
7,2010,1,2,07062015,1105,812,120040,22051998,417,2,...,,,,,,,,,2015,AC
8,2108,1,2,16062015,2330,812,120020,16121972,442,2,...,,M,,,,,,,2015,AC
9,2115,1,2,24062015,1630,812,120070,15041973,442,2,...,,M,,,,,,,2015,AC


In [16]:
print(violencia_feminina[["DTOBITO", "ESTADO", "ANO", "CAUSABAS"]].sample(10))

        DTOBITO ESTADO   ANO  CAUSABAS
32954  26092021     SP  2021      X999
3050   14122017     BA  2017  X990    
6300   17032018     CE  2018      X950
25069  30042022     RJ  2022      X919
15334  16042018     MT  2018      Y047
17889  05012016     PB  2016  X999    
22907  16042015     RJ  2015  X940    
7101   06052020     CE  2020      X954
22814  24102015     RJ  2015  X910    
28859  01012015     SC  2015  X990    


In [17]:
violencia_feminina.groupby(["ESTADO", "ANO"]).size()

ESTADO  ANO 
AC      2015    19
        2016    23
        2017    34
        2018    35
        2019    32
                ..
TO      2018    46
        2019    32
        2020    36
        2021    37
        2022    29
Length: 216, dtype: int64

In [18]:
print(sorted(violencia_feminina["ANO"].unique()))


[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
